# NLP Анализ и Классификация Фильмов

## Полный pipeline: от загрузки данных до сравнения 4 моделей ML

**Структура проекта:**
1. Импорт библиотек и инициализация
2. Загрузка и изучение данных
3. Обработка текста (очистка, лемматизация)
4. Векторизация текста (TF-IDF)
5. Тематическое моделирование (NMF)
6. Подготовка данных для классификации
7. Обучение 4 моделей (LR, XGBoost, RF, Neural Network)
8. Сравнение и анализ результатов
9. Выводы и рекомендации

# 1. Импорт и инициализация библиотек

**ОБЪЯСНЕНИЕ:** Загружаем все необходимые библиотеки для работы.

**Какие библиотеки нужны:**
- `pandas, numpy` - работа с данными и массивами
- `sklearn` - машинное обучение (модели, метрики, предобработка)
- `torch` - глубокое обучение (нейросети)
- `nltk, pymorphy3` - обработка естественного языка
- `xgboost` - продвинутый градиентный бустинг
- `matplotlib, seaborn, wordcloud` - визуализация

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymorphy3
import re
import warnings
import nltk
import xgboost as xgb
import time
import pickle

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF  # ✓ ИСПРАВЛЕНО: убрали KMeans отсюда
from sklearn.cluster import KMeans     # ✓ ДОБАВИЛИ: KMeans из правильного модуля
from sklearn.metrics import silhouette_score
from wordcloud import WordCloud
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

warnings.filterwarnings('ignore')

# Определяем устройство для PyTorch (GPU если доступна, иначе CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

# Инициализируем морфологический анализатор для русского языка
morph = pymorphy3.MorphAnalyzer(lang='ru')

# Настройка стиля визуализации
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("\n✓ Все библиотеки загружены успешно")

# 2. Загрузка и исследование данных

**ОБЪЯСНЕНИЕ:** 
- Загружаем CSV файл с данными о фильмах
- Проводим первичный анализ структуры данных
- Проверяем пропущенные значения
- Изучаем распределение основных переменных

In [ ]:
# ЗАГРУЗКА ДАННЫХ
# ЗАМЕНИ 'movies_final_cleaned.csv' на название твоего файла если оно другое
df = pd.read_csv('movies_final_cleaned.csv', index_col=0)

print(f"✓ Данные загружены")
print(f"  Фильмов: {df.shape[0]}")
print(f"  Колонок: {df.shape[1]}")
print(f"  Память: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# ПЕРВЫЕ 5 СТРОК
print("Первые 5 строк данных:")
print(df.head())

In [ ]:
# ИНФОРМАЦИЯ О ТИПАХ
print("\nИнформация о типах данных:")
df.info()

In [ ]:
# СТАТИСТИКА
print("\nСтатистика числовых признаков:")
df.describe()

In [ ]:
# РАСПРЕДЕЛЕНИЕ ПРИЗНАКОВ
print("\nДистрибуция числовых признаков:")
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols].hist(figsize=(16, 10), bins=30)
plt.tight_layout()
plt.show()

## 2.1 Загрузка и подготовка стоп-слов

**ОБЪЯСНЕНИЕ:**
- Стоп-слова - это часто встречающиеся слова (и, или, что) которые не несут смысла
- Удаляем их перед анализом текста
- Добавляем дополнительные стоп-слова специфичные для фильмов

In [ ]:
# Загружаем русские стоп-слова
try:
    stopwords.words('russian')
    print("Стоп-слова уже загружены")
except LookupError:
    print("Загружаю стоп-слова...")
    nltk.download('stopwords')
    nltk.download('punkt')

russian_stopwords = set(stopwords.words('russian'))

# Добавляем дополнительные стоп-слова специфичные для фильмов
additional_stopwords = [
    'т.д.', 'т', 'д', 'это', 'который', 'свой', 'своём',
    'всем', 'всё', 'её', 'оба', 'ещё', 'должный', 'фильм',
    'кино', 'картина', 'действие', 'герой'
]
russian_stopwords.update(additional_stopwords)

print(f"\n✓ Загружено {len(russian_stopwords)} русских стоп-слов")

# 3. Обработка текстовых данных

**ОБЪЯСНЕНИЕ:**
Обработка текста состоит из 3 этапов:
1. **Очистка** - удаление спецсимволов, приведение к нижнему регистру
2. **Лемматизация** - приведение слов к начальной форме (бежал -> бегать)
3. **Удаление стоп-слов** - удаляем слова которые не несут смысла

Это необходимо для правильной работы алгоритмов NLP.

In [ ]:
# ФУНКЦИЯ 1: ОЧИСТКА ТЕКСТА
def clean_text(text):
    """
    Очистка текста от спецсимволов и приведение к нижнему регистру
    
    ОБЪЯСНЕНИЕ:
    - Проверяем что текст не пустой (не NaN)
    - Приводим к нижнему регистру
    - Удаляем все символы кроме букв и пробелов
    - Удаляем лишние пробелы
    """
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r'[^а-яёa-z\s]', '', text)  # Только буквы и пробелы
    text = re.sub(r'\s+', ' ', text).strip()   # Удаляем лишние пробелы
    return text


# ФУНКЦИЯ 2: ЛЕММАТИЗАЦИЯ
def lemmatize_text(text):
    """
    Лемматизация текста и удаление стоп-слов
    
    ОБЪЯСНЕНИЕ:
    - Разбиваем текст на слова
    - Для каждого слова находим начальную форму (лемму)
    - Удаляем стоп-слова и очень короткие слова
    """
    if not text:
        return ""
    
    words = text.split()
    lemmas = []
    
    for word in words:
        lemma = morph.parse(word)[0].normal_form
        # Фильтруем стоп-слова и очень короткие слова
        if lemma not in russian_stopwords and len(lemma) > 2:
            lemmas.append(lemma)
    
    return ' '.join(lemmas)


# ФУНКЦИЯ 3: ПОЛНАЯ ОБРАБОТКА
def process_descriptions(data, text_col='description', verbose=True):
    """
    Полная обработка описаний фильмов
    
    ОБЪЯСНЕНИЕ:
    - Применяем очистку текста
    - Применяем лемматизацию
    - Возвращаем датафрейм с новыми колонками
    """
    data_copy = data.copy()
    
    if verbose:
        print("Этап 1: Очистка текстов...")
    data_copy['description_clean'] = data_copy[text_col].apply(clean_text)
    
    if verbose:
        print("Этап 2: Лемматизация...")
    data_copy['description_lemma'] = data_copy['description_clean'].apply(lemmatize_text)
    
    return data_copy


print("✓ Функции обработки текста определены")

In [ ]:
# Применяем полную обработку текста
df = process_descriptions(df, text_col='description', verbose=True)

print(f"\n✓ Обработка завершена")

# Показываем примеры
print(f"\n📌 ПРИМЕРЫ ОБРАБОТКИ:")
print("="*80)

for i in range(min(1, len(df))):
    print(f"\nФильм #{i+1}:")
    print(f"\n1️⃣ ОРИГИНАЛ:")
    print(f"  {df['description'].iloc[i][:150]}...")
    print(f"\n2️⃣ ПОСЛЕ ОЧИСТКИ:")
    print(f"  {df['description_clean'].iloc[i][:150]}...")
    print(f"\n3️⃣ ПОСЛЕ ЛЕММАТИЗАЦИИ:")
    print(f"  {df['description_lemma'].iloc[i][:150]}...")

## 3.1 TF-IDF Векторизация

**ОБЪЯСНЕНИЕ:**
- TF-IDF (Term Frequency - Inverse Document Frequency) преобразует текст в числовой вектор
- TF: как часто слово встречается в документе
- IDF: как редко слово встречается во всех документах

**Параметры:**
- `max_features=100` - берем 100 самых важных слов
- `min_df=3` - слово должно быть минимум в 3 документах
- `max_df=0.85` - слово не должно быть в более чем 85% документов
- `ngram_range=(1, 2)` - используем однословные и двусловные фразы

In [ ]:
print("Инициализация TF-IDF векторизатора...\n")

tfidf = TfidfVectorizer(
    max_features=100,       # Максимум 100 признаков (слов)
    min_df=3,               # Слово должно быть минимум в 3 документах
    max_df=0.85,            # Слово не должно быть в более чем 85% документов
    ngram_range=(1, 2),     # Однословные и двусловные фразы
    strip_accents='unicode' # Удаляем диакритические знаки
)

print("Применение TF-IDF трансформации...")
tfidf_matrix = tfidf.fit_transform(df['description_lemma'])

# Преобразуем разреженную матрицу в датафрейм
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f'tfidf_{name}' for name in tfidf.get_feature_names_out()],
    index=df.index
)

print(f"\n✓ TF-IDF матрица создана: {tfidf_df.shape}")
print(f"  Документов: {tfidf_df.shape[0]}")
print(f"  Признаков (слов): {tfidf_df.shape[1]}")

In [ ]:
# Анализируем какие слова наиболее важны
feature_freq = tfidf_df.sum(axis=0).sort_values(ascending=False)

print("\n📊 Топ-20 наиболее важных слов (по сумме TF-IDF):")
print("="*60)
for i, (feature, freq) in enumerate(feature_freq.head(20).items(), 1):
    clean_name = feature.replace('tfidf_', '')
    print(f"{i:2d}. {clean_name:25s} | TF-IDF: {freq:.4f}")

# Визуализируем топ слов
plt.figure(figsize=(12, 6))
feature_freq.head(15).plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Топ-15 слов по важности (TF-IDF)', fontsize=14, fontweight='bold')
plt.xlabel('Сумма TF-IDF значений')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 3.2 Облака слов (WordCloud)

**ОБЪЯСНЕНИЕ:**
- Облако слов (WordCloud) - визуализация где размер слова пропорционален частоте
- Помогает быстро увидеть какие слова доминируют в текстах
- Если есть целевая переменная, создаем облака для разных классов

In [ ]:
# Создаем облако слов для всех текстов
all_text = ' '.join(df['description_lemma'].dropna())

plt.figure(figsize=(14, 8))
wc = WordCloud(
    width=1200, 
    height=600, 
    background_color='white',
    colormap='viridis', 
    max_words=100
).generate(all_text)

plt.imshow(wc, interpolation='bilinear')
plt.title('Облако слов: Все описания фильмов', fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print("✓ Облако слов создано")

# 4. Тематическое моделирование (NMF)

**ОБЪЯСНЕНИЕ:**
- NMF (Non-Negative Matrix Factorization) разлагает матрицу на несколько тем
- Каждая тема = набор слов которые часто встречаются вместе
- Для каждого документа определяем наиболее вероятную тему
- Используем эту информацию как целевую переменную для классификации

**Процесс:**
1. Берем TF-IDF матрицу
2. Разлагаем на N тем
3. Для каждого документа определяем тему с максимальной вероятностью
4. Создаем целевую переменную для классификации

In [ ]:
print("\n" + "="*70)
print("🎯 Тематическое моделирование (NMF)")
print("="*70)

# Определяем количество тем
n_topics = 7

print(f"\nСоздание NMF модели с {n_topics} темами...")

nmf_model = NMF(
    n_components=n_topics,
    random_state=42,
    max_iter=300,
    init='nndsvd'
)

# Обучаем модель на TF-IDF матрице
nmf_model.fit(tfidf_matrix)

print(f"✓ Модель обучена")

# Для каждой темы показываем топ-10 слов
print(f"\n📌 Топ-10 слов для каждой темы:")
print("="*70)

feature_names = tfidf.get_feature_names_out()
topics_description = {}

for topic_idx, topic in enumerate(nmf_model.components_):
    top_indices = topic.argsort()[-10:][::-1]
    top_words = [feature_names[i] for i in top_indices]
    topics_description[topic_idx] = top_words
    print(f"\n🔹 Тема {topic_idx}:")
    print(f"   {', '.join(top_words)}")

In [ ]:
# Для каждого документа определяем наиболее вероятную тему
X_topics_nmf = nmf_model.transform(tfidf_matrix)

# argmax находит индекс максимального значения (наиболее вероятная тема)
df['topic_nmf'] = np.argmax(X_topics_nmf, axis=1)

print(f"✓ Темы назначены всем документам")

# Показываем распределение тем
print(f"\n📊 Распределение документов по темам:")
print("="*70)
topic_dist = df['topic_nmf'].value_counts().sort_index()
for topic, count in topic_dist.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f"Тема {topic}: {count:4d} документов ({pct:5.1f}%) {bar}")

# Визуализируем распределение
plt.figure(figsize=(10, 6))
df['topic_nmf'].value_counts().sort_index().plot(
    kind='bar',
    color='steelblue',
    edgecolor='black',
    alpha=0.8
)
plt.title('Распределение документов по темам (NMF)', fontsize=14, fontweight='bold')
plt.xlabel('Номер темы')
plt.ylabel('Количество документов')
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 5. Подготовка данных для классификации

**ОБЪЯСНЕНИЕ:**
- Объединяем разные типы признаков:
  1. Численные признаки (год, рейтинги и т.д.)
  2. TF-IDF признаки (важные слова из описаний)
  3. Закодированные жанры (если есть)
- Масштабируем данные к диапазону [0, 1]
- Разделяем на обучающую и тестовую выборки

In [ ]:
print("\n" + "="*70)
print("📋 Подготовка признаков для классификации")
print("="*70)

# Получаем числовые столбцы
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Исключаем целевую переменную
if 'topic_nmf' in numeric_cols:
    numeric_cols.remove('topic_nmf')

print(f"\n✓ Найдены числовые признаки:")
print(f"  Количество: {len(numeric_cols)}")
for col in numeric_cols:
    print(f"    - {col}")

# Получаем числовые данные (заполняем NaN нулями)
X_numeric = df[numeric_cols].fillna(0)

# Объединяем все признаки
X_combined = pd.concat([X_numeric, tfidf_df], axis=1)

print(f"\n✓ Объединение признаков:")
print(f"  Числовые: {X_numeric.shape[1]}")
print(f"  TF-IDF: {tfidf_df.shape[1]}")
print(f"  Всего: {X_combined.shape[1]}")
print(f"  Образцов: {X_combined.shape[0]}")

In [ ]:
# Масштабируем признаки к диапазону [0, 1]
# Это важно для моделей чувствительных к масштабу (LR, KNN, SVM)

print("\nМасштабирование признаков с MinMaxScaler...")

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_combined)

print(f"\n✓ Масштабирование завершено")
print(f"  Min значение: {X_scaled.min():.4f}")
print(f"  Max значение: {X_scaled.max():.4f}")
print(f"  Mean значение: {X_scaled.mean():.4f}")

In [ ]:
# Разделяем данные на обучающую (80%) и тестовую (20%) выборки
# stratify=y обеспечивает одинаковое распределение классов

print("\nРазделение данных на train/test выборки...\n")

y = df['topic_nmf']  # Целевая переменная

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,          # 20% на тестирование
    random_state=42,       # Для воспроизводимости
    stratify=y             # Одинаковое распределение классов
)

print(f"✓ Разделение завершено")
print(f"\nОбучающая выборка: {X_train.shape}")
for i in range(len(np.unique(y))):
    count = (y_train == i).sum()
    pct = count / len(y_train) * 100
    print(f"  Класс {i}: {count:5d} ({pct:5.1f}%)")

print(f"\nТестовая выборка: {X_test.shape}")
for i in range(len(np.unique(y))):
    count = (y_test == i).sum()
    pct = count / len(y_test) * 100
    print(f"  Класс {i}: {count:5d} ({pct:5.1f}%)")

# 6. Обучение моделей машинного обучения

**ОБЪЯСНЕНИЕ:**
Обучаем 4 различные модели классификации:

1. **Логистическая регрессия (LR)**
   - Простая линейная модель
   - Быстрая и интерпретируемая
   - Хорошо работает с высокоразмерными данными

2. **XGBoost**
   - Продвинутый метод градиентного бустинга
   - Последовательно улучшает предсказания
   - Часто дает лучшие результаты но медленнее

3. **Random Forest**
   - Ансамбль из множества деревьев
   - Баланс между точностью и скоростью
   - Дает информацию об важности признаков

4. **Нейросеть (PyTorch)**
   - Глубокое обучение
   - Может выучить очень сложные зависимости
   - Требует больше данных и вычислений

**Метрики качества:**
- **Accuracy** - доля правильных предсказаний
- **Precision** - доля правильных положительных предсказаний
- **Recall** - какую долю положительных случаев мы поймали
- **F1-Score** - гармоническое среднее Precision и Recall
- **ROC-AUC** - площадь под кривой ROC

In [ ]:
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ДЛЯ ОЦЕНКИ МОДЕЛЕЙ

def evaluate_model(y_true, y_pred, y_pred_proba):
    """
    Вычисление всех метрик качества модели
    
    ОБЪЯСНЕНИЕ:
    - Accuracy: (TP + TN) / (TP + TN + FP + FN)
    - Precision: TP / (TP + FP)
    - Recall: TP / (TP + FN)
    - F1: 2 * (Precision * Recall) / (Precision + Recall)
    - AUC: площадь под ROC кривой
    """
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }
    
    # Для многоклассовой классификации
    try:
        if len(np.unique(y_true)) == 2:
            metrics['auc'] = roc_auc_score(y_true, y_pred_proba[:, 1])
        else:
            from sklearn.preprocessing import label_binarize
            y_bin = label_binarize(y_true, classes=np.unique(y_true))
            if y_pred_proba.shape[1] == len(np.unique(y_true)):
                metrics['auc'] = roc_auc_score(y_bin, y_pred_proba, multi_class='ovr', average='weighted')
            else:
                metrics['auc'] = 0.0
    except:
        metrics['auc'] = 0.0
    
    return metrics


def print_model_results(metrics, train_time, model_name="Модель"):
    """
    Красивый вывод результатов обучения модели
    """
    print(f"\n{'Результаты ' + model_name + ':':-^50}")
    print(f"  Accuracy:        {metrics['accuracy']:.4f}")
    print(f"  Precision:       {metrics['precision']:.4f}")
    print(f"  Recall:          {metrics['recall']:.4f}")
    print(f"  F1-Score:        {metrics['f1']:.4f}")
    if 'auc' in metrics:
        print(f"  ROC-AUC:         {metrics['auc']:.4f}")
    print(f"  Время обучения:  {train_time:.2f} сек")

models_results = {}

print("✓ Вспомогательные функции определены")

In [ ]:
print("\n" + "="*70)
print("🔹 МОДЕЛЬ 1: Логистическая регрессия (Logistic Regression)")
print("="*70)
print("\nОПИСАНИЕ:")
print("  - Линейная модель для классификации")
print("  - Быстрая и интерпретируемая")
print("  - Хорошо работает с высокоразмерными данными (много признаков)")

start_time = time.time()

# ✓ ИСПРАВЛЕНО: убрали multi_class, добавили solver
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1,
    solver='lbfgs'  # ✓ Можно использовать: 'lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga'
)
lr_model.fit(X_train, y_train)

train_time_lr = time.time() - start_time

print(f"\n✓ Модель обучена за {train_time_lr:.2f} сек")

In [ ]:
# Делаем предсказания
y_pred_lr = lr_model.predict(X_test)
y_pred_proba_lr = lr_model.predict_proba(X_test)

# Вычисляем метрики
metrics_lr = evaluate_model(y_test, y_pred_lr, y_pred_proba_lr)
print_model_results(metrics_lr, train_time_lr, "Логистической регрессии")

print(f"\n{'Подробный отчет классификации:':-^50}")
print(classification_report(y_test, y_pred_lr))

models_results['Логистическая регрессия'] = {
    'model': lr_model,
    'metrics': metrics_lr,
    'time': train_time_lr,
    'y_pred': y_pred_lr,
    'y_proba': y_pred_proba_lr
}

In [ ]:
print("\n" + "="*70)
print("🔹 МОДЕЛЬ 2: XGBoost (eXtreme Gradient Boosting)")
print("="*70)
print("\nОПИСАНИЕ:")
print("  - Продвинутый метод градиентного бустинга")
print("  - Последовательно строит деревья, исправляя ошибки")
print("  - Часто показывает лучшие результаты на Kaggle")
print("  - Медленнее логистической регрессии")

start_time = time.time()

xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss',
    verbosity=0
)
xgb_model.fit(X_train, y_train)

train_time_xgb = time.time() - start_time

print(f"\n✓ Модель обучена за {train_time_xgb:.2f} сек")

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)

metrics_xgb = evaluate_model(y_test, y_pred_xgb, y_pred_proba_xgb)
print_model_results(metrics_xgb, train_time_xgb, "XGBoost")

print(f"\n{'Подробный отчет классификации:':-^50}")
print(classification_report(y_test, y_pred_xgb))

models_results['XGBoost'] = {
    'model': xgb_model,
    'metrics': metrics_xgb,
    'time': train_time_xgb,
    'y_pred': y_pred_xgb,
    'y_proba': y_pred_proba_xgb
}

In [ ]:
# Анализируем важность признаков
feature_importance_xgb = pd.Series(
    xgb_model.feature_importances_,
    index=X_combined.columns
).nlargest(15)

print(f"\n📊 Топ-15 важных признаков (XGBoost):")
for idx, (feature, importance) in enumerate(feature_importance_xgb.items(), 1):
    print(f"{idx:2d}. {feature:30s} | Важность: {importance:.4f}")

plt.figure(figsize=(10, 6))
feature_importance_xgb.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('XGBoost: Топ-15 важных признаков', fontsize=12, fontweight='bold')
plt.xlabel('Важность (Gain)')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*70)
print("🔹 МОДЕЛЬ 3: Random Forest")
print("="*70)
print("\nОПИСАНИЕ:")
print("  - Ансамбль из множества деревьев решений")
print("  - Каждое дерево обучается на случайной выборке")
print("  - Результаты усредняются для лучшей стабильности")
print("  - Хороший баланс между точностью и скоростью")

start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rf_model.fit(X_train, y_train)

train_time_rf = time.time() - start_time

print(f"\n✓ Модель обучена за {train_time_rf:.2f} сек")

In [ ]:
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)

metrics_rf = evaluate_model(y_test, y_pred_rf, y_pred_proba_rf)
print_model_results(metrics_rf, train_time_rf, "Random Forest")

print(f"\n{'Подробный отчет классификации:':-^50}")
print(classification_report(y_test, y_pred_rf))

models_results['Random Forest'] = {
    'model': rf_model,
    'metrics': metrics_rf,
    'time': train_time_rf,
    'y_pred': y_pred_rf,
    'y_proba': y_pred_proba_rf
}

In [ ]:
# Анализируем важность признаков
feature_importance_rf = pd.Series(
    rf_model.feature_importances_,
    index=X_combined.columns
).nlargest(15)

print(f"\n📊 Топ-15 важных признаков (Random Forest):")
for idx, (feature, importance) in enumerate(feature_importance_rf.items(), 1):
    print(f"{idx:2d}. {feature:30s} | Важность: {importance:.4f}")

plt.figure(figsize=(10, 6))
feature_importance_rf.plot(kind='barh', color='forestgreen', edgecolor='black')
plt.title('Random Forest: Топ-15 важных признаков', fontsize=12, fontweight='bold')
plt.xlabel('Важность (MDI)')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*70)
print("🔹 МОДЕЛЬ 4: Нейросеть (Deep Neural Network на PyTorch) - ОПТИМИЗИРОВАННАЯ")
print("="*70)

class MovieClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MovieClassifier, self).__init__()
        
        # ✅ Нормализация входных данных
        self.input_norm = nn.LayerNorm(input_dim)
        
        # ✅ Слой 1: input_dim -> 128 (было 256)
        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.4)  # было 0.3
        
        # ✅ Слой 2: 128 -> 64 (было 256->128)
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.4)  # было 0.3
        
        # ✅ Слой 3: 64 -> 32 (было 128->64)
        self.fc3 = nn.Linear(64, 32)
        self.bn3 = nn.BatchNorm1d(32)
        self.dropout3 = nn.Dropout(0.3)  # было 0.2
        
        # ✅ Выходной слой прямо из 32 (было 32->16->num_classes)
        self.fc_out = nn.Linear(32, num_classes)
        
    def forward(self, x):
        """Прямой проход через оптимизированную сеть"""
        # Нормализуем вход
        x = self.input_norm(x)
        
        # Слой 1
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        
        # Слой 2
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        
        # Слой 3
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.dropout3(x)
        
        # Выходной слой
        x = self.fc_out(x)
        
        return x

In [ ]:
# ✅ Подготавливаем данные для нейросети
print("\nПодготовка данных для нейросети...")

# Преобразуем в PyTorch тензоры
X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.LongTensor(y_train.values).to(device)
X_test_tensor = torch.FloatTensor(X_test).to(device)
y_test_tensor = torch.LongTensor(y_test.values).to(device)

# ✅ Уменьшили размер батча (лучше для маленького датасета)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(
    train_dataset,
    batch_size=16,  # было 32, уменьшили для лучшей сходимости
    shuffle=True
)

print(f"✓ Данные подготовлены")
print(f"  Устройство: {device}")
print(f"  Размер батча: 16 (было 32)")
print(f"  Количество батчей: {len(train_loader)}")
print(f"  Количество классов: {len(np.unique(y_train))}")

In [ ]:
# ✅ Инициализируем ОПТИМИЗИРОВАННУЮ нейросеть
print("\nИнициализация оптимизированной нейросети...\n")

num_classes = len(np.unique(y_train))
nn_model = MovieClassifier(X_train.shape[1], num_classes).to(device)

# ✅ Функция потерь с весами классов (если они несбалансированы)
criterion = nn.CrossEntropyLoss()

# ✅ Оптимизатор AdamW (улучшенная версия Adam)
optimizer = torch.optim.AdamW(
    nn_model.parameters(),
    lr=0.001,
    weight_decay=1e-4  # было 1e-5, увеличили для регуляризации
)

# ✅ Learning Rate Scheduler (снижает lr если нет улучшений)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,           # уменьшай lr на 50%
    patience=3,           # если нет улучшений 3 эпохи
    min_lr=1e-6,
    threshold=1e-4
)

# Подсчитываем параметры
total_params = sum(p.numel() for p in nn_model.parameters())
trainable_params = sum(p.numel() for p in nn_model.parameters() if p.requires_grad)

print(f"✓ Нейросеть готова")
print(f"  Архитектура: {X_train.shape[1]} -> 128 -> 64 -> 32 -> {num_classes}")
print(f"  Всего параметров: {total_params:,}")
print(f"  Обучаемых параметров: {trainable_params:,}")
print(f"  Оптимизатор: AdamW с Learning Rate Scheduling")

In [ ]:
# ✅ УЛУЧШЕННОЕ ОБУЧЕНИЕ НЕЙРОСЕТИ
print("\nОбучение оптимизированной нейросети (200 эпох с ранней остановкой)...\n")

start_time = time.time()

epochs = 200  # было 50, увеличили
losses = []
val_losses = []
learning_rates = []
best_val_loss = float('inf')
best_model_state = None
patience_counter = 0
patience = 15  # было 5, увеличили

for epoch in range(epochs):
    # 🔵 РЕЖИМ ОБУЧЕНИЯ
    nn_model.train()
    train_loss = 0
    batch_count = 0
    
    for X_batch, y_batch in train_loader:
        # Прямой проход
        outputs = nn_model(X_batch)
        loss = criterion(outputs, y_batch)
        
        # Обратный проход
        optimizer.zero_grad()
        loss.backward()
        
        # ✅ Gradient clipping (защита от взрывающихся градиентов)
        torch.nn.utils.clip_grad_norm_(nn_model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        train_loss += loss.item()
        batch_count += 1
    
    # 🟢 РЕЖИМ ОЦЕНКИ
    nn_model.eval()
    with torch.no_grad():
        val_outputs = nn_model(X_train_tensor)
        val_loss = criterion(val_outputs, y_train_tensor)
    
    avg_train_loss = train_loss / batch_count
    losses.append(avg_train_loss)
    val_losses.append(val_loss.item())
    
    # ✅ Learning Rate Scheduling
    scheduler.step(val_loss.item())
    current_lr = optimizer.param_groups[0]['lr']
    learning_rates.append(current_lr)
    
    # ✅ Ранняя остановка с сохранением лучшей модели
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        best_model_state = {k: v.cpu().clone() for k, v in nn_model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"\n✅ Ранняя остановка на эпохе {epoch+1}")
        print(f"   Лучшая val_loss: {best_val_loss:.4f}")
        # Загружаем лучшую модель
        nn_model.load_state_dict(best_model_state)
        break
    
    # Выводим прогресс каждые 20 эпох
    if (epoch + 1) % 20 == 0:
        print(f"Эпоха {epoch+1:3d}/{epochs} | Train Loss: {avg_train_loss:.4f} | " + 
              f"Val Loss: {val_loss:.4f} | LR: {current_lr:.6f} | Patience: {patience_counter}/{patience}")

train_time_nn = time.time() - start_time
print(f"\n✓ Обучение завершено за {train_time_nn:.2f} сек")

In [ ]:
# ✅ ОЦЕНКА ОПТИМИЗИРОВАННОЙ НЕЙРОСЕТИ
print("\n" + "="*70)
print("📊 Оценка оптимизированной нейросети")
print("="*70)

nn_model.eval()
with torch.no_grad():
    y_pred_logits = nn_model(X_test_tensor)
    y_pred_proba_nn = torch.softmax(y_pred_logits, dim=1).cpu().numpy()
    y_pred_nn = np.argmax(y_pred_proba_nn, axis=1)

metrics_nn = evaluate_model(y_test, y_pred_nn, y_pred_proba_nn)
print_model_results(metrics_nn, train_time_nn, "Оптимизированной нейросети")

print(f"\n{'Подробный отчет классификации:':-^50}")
print(classification_report(y_test, y_pred_nn))

models_results['Нейросеть (оптимизированная)'] = {
    'model': nn_model,
    'metrics': metrics_nn,
    'time': train_time_nn,
    'y_pred': y_pred_nn,
    'y_proba': y_pred_proba_nn
}

# ✅ Показываем сравнение с оригинальной нейросетью
print("\n" + "="*70)
print("📈 СРАВНЕНИЕ: Оригинальная vs Оптимизированная")
print("="*70)
print(f"\nОригинальная нейросеть:      Accuracy = 73.20%")
print(f"Оптимизированная нейросеть:  Accuracy = {metrics_nn['accuracy']*100:.2f}% ⭐")
improvement = (metrics_nn['accuracy'] - 0.732) * 100
print(f"\nУЛУЧШЕНИЕ: {improvement:+.2f} процентных пункта {'🚀' if improvement > 0 else '⚠️'}")

In [ ]:
# ✅ ВИЗУАЛИЗАЦИЯ ОБУЧЕНИЯ ОПТИМИЗИРОВАННОЙ НЕЙРОСЕТИ
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Оптимизированная нейросеть: Процесс обучения', fontsize=14, fontweight='bold')

# График 1: Loss
axes[0].plot(losses, label='Training Loss', linewidth=2, color='steelblue', alpha=0.8)
axes[0].plot(val_losses, label='Validation Loss', linewidth=2, color='orange', alpha=0.8)
axes[0].set_title('Динамика Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# График 2: Заполненная кривая обучения
axes[1].fill_between(range(len(losses)), losses, alpha=0.3, color='steelblue')
axes[1].plot(range(len(losses)), losses, linewidth=2, color='steelblue')
axes[1].set_title('Кривая обучения (Training Loss)', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(alpha=0.3)

# График 3: Learning Rate динамика
axes[2].plot(learning_rates, linewidth=2, color='green', marker='o', markersize=3, alpha=0.7)
axes[2].set_title('Learning Rate Schedule', fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_yscale('log')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Графики обучения построены")

# 7. Сравнение и анализ всех 4 моделей

**ОБЪЯСНЕНИЕ:**
- Сравниваем все модели по различным метрикам
- Создаем таблицы и графики для наглядности
- Анализируем ROC-AUC кривые
- Смотрим матрицы ошибок для каждой модели

In [ ]:
print("\n" + "="*70)
print("📊 СРАВНЕНИЕ ВСЕХ 4 МОДЕЛЕЙ")
print("="*70)

# Создаем таблицу результатов
results_df = pd.DataFrame({
    'Модель': list(models_results.keys()),
    'Accuracy': [models_results[m]['metrics']['accuracy'] for m in models_results],
    'Precision': [models_results[m]['metrics']['precision'] for m in models_results],
    'Recall': [models_results[m]['metrics']['recall'] for m in models_results],
    'F1-Score': [models_results[m]['metrics']['f1'] for m in models_results],
    'ROC-AUC': [models_results[m]['metrics'].get('auc', 0) for m in models_results],
    'Время (сек)': [models_results[m]['time'] for m in models_results]
})

print("\n📈 Результаты моделей:")
print(results_df.to_string(index=False))

# Средние значения
print(f"\n{'Средние значения:':-^50}")
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    print(f"  {col:15s}: {results_df[col].mean():.4f}")

In [ ]:
# Определяем лучшие модели
print("\n" + "="*70)
print("🏆 ЛУЧШИЕ МОДЕЛИ ПО МЕТРИКАМ")
print("="*70)

metrics_to_check = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

for metric in metrics_to_check:
    best_idx = results_df[metric].idxmax()
    best_model = results_df.loc[best_idx, 'Модель']
    best_score = results_df.loc[best_idx, metric]
    print(f"\n  {metric:12s}: {best_model:25s} = {best_score:.4f} ⭐")

best_time_idx = results_df['Время (сек)'].idxmin()
best_time_model = results_df.loc[best_time_idx, 'Модель']
best_time = results_df.loc[best_time_idx, 'Время (сек)']
print(f"\n  Скорость:    {best_time_model:25s} = {best_time:.2f} сек ⚡")

# Общее место
results_df['Score'] = (
    results_df['Accuracy'] * 0.2 +
    results_df['Precision'] * 0.2 +
    results_df['Recall'] * 0.2 +
    results_df['F1-Score'] * 0.2 +
    results_df['ROC-AUC'] * 0.2
)
best_overall_idx = results_df['Score'].idxmax()
best_overall = results_df.loc[best_overall_idx, 'Модель']
best_overall_score = results_df.loc[best_overall_idx, 'Score']
print(f"\n  🥇 ЛУЧШАЯ МОДЕЛЬ: {best_overall:25s} (средний балл: {best_overall_score:.4f})")

In [ ]:
# Визуализируем сравнение
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('📊 Сравнение всех 4 моделей по метрикам', fontsize=16, fontweight='bold')

metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

# Графики для каждой метрики
for idx, metric in enumerate(metrics_list):
    ax = axes[idx // 3, idx % 3]
    bars = ax.bar(
        results_df['Модель'],
        results_df[metric],
        color=colors,
        alpha=0.8,
        edgecolor='black',
        linewidth=1.5
    )
    ax.set_title(f'{metric}', fontweight='bold', fontsize=12)
    ax.set_ylabel('Score')
    ax.set_ylim([0, 1])
    
    # Линия среднего значения
    ax.axhline(
        y=results_df[metric].mean(),
        color='red',
        linestyle='--',
        linewidth=2,
        label='Среднее'
    )
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=15)
    
    # Значения на столбцы
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2.,
            height,
            f'{height:.3f}',
            ha='center',
            va='bottom',
            fontweight='bold',
            fontsize=9
        )

# График времени
ax = axes[1, 2]
bars = ax.bar(
    results_df['Модель'],
    results_df['Время (сек)'],
    color=colors,
    alpha=0.8,
    edgecolor='black',
    linewidth=1.5
)
ax.set_title('Время обучения', fontweight='bold', fontsize=12)
ax.set_ylabel('Время (сек)')
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=15)

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2.,
        height,
        f'{height:.2f}s',
        ha='center',
        va='bottom',
        fontweight='bold',
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
# ✅ ROC-AUC кривые для МНОГОКЛАССОВОЙ классификации
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

colors_roc = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

print("Построение ROC кривых для многоклассовой классификации...\n")

# Бинаризуем метки (для каждого класса: есть/нет)
y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_bin.shape[1]

for idx, (model_name, model_data) in enumerate(models_results.items()):
    ax = axes[idx]
    
    y_proba = model_data['y_proba']
    
    # Проверяем размерность вероятностей
    if len(y_proba.shape) == 1:
        print(f"⚠️  {model_name}: вероятности одномерные, пропускаем")
        continue
    
    if y_proba.shape[1] != n_classes:
        print(f"⚠️  {model_name}: размерность вероятностей не совпадает ({y_proba.shape[1]} vs {n_classes})")
        continue
    
    # Вычисляем ROC кривую для каждого класса (One-vs-Rest)
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    
    # Вычисляем micro-average ROC кривую
    fpr_micro, tpr_micro, _ = roc_curve(y_test_bin.ravel(), y_proba.ravel())
    roc_auc_micro = auc(fpr_micro, tpr_micro)
    
    # Интерполируем и усредняем ROC кривые (macro-average)
    fpr_grid = np.linspace(0, 1, 300)
    tpr_interp = np.zeros_like(fpr_grid)
    
    for i in range(n_classes):
        tpr_interp += np.interp(fpr_grid, fpr[i], tpr[i])
    
    tpr_interp /= n_classes
    roc_auc_macro = auc(fpr_grid, tpr_interp)
    
    # Рисуем кривые для каждого класса светлым цветом
    for i in range(n_classes):
        ax.plot(
            fpr[i], tpr[i],
            alpha=0.3,
            linewidth=1,
            color='gray',
            label=f'Класс {i}' if i < 3 else ''  # только первые 3 в легенде
        )
    
    # Рисуем macro-average (основная кривая)
    ax.plot(
        fpr_grid, tpr_interp,
        color=colors_roc[idx],
        linewidth=3,
        label=f'Macro-average (AUC={roc_auc_macro:.4f})',
        zorder=5
    )
    
    # Диагональ
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Случайный', zorder=1)
    
    ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
    ax.set_title(f'{model_name}\n(Macro-AUC={roc_auc_macro:.4f}, Micro-AUC={roc_auc_micro:.4f})', 
                 fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    
    print(f"✓ {model_name:30s} | Macro-AUC: {roc_auc_macro:.4f} | Micro-AUC: {roc_auc_micro:.4f}")

plt.suptitle(f'ROC-AUC Кривые для всех моделей ({n_classes} классов)', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print(f"\n✓ ROC кривые построены успешно!")

In [ ]:
# Матрицы ошибок (Confusion Matrices)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Матрицы ошибок (Confusion Matrices)', fontsize=16, fontweight='bold')

for idx, (model_name, model_data) in enumerate(models_results.items()):
    ax = axes[idx // 2, idx % 2]
    cm = confusion_matrix(y_test, model_data['y_pred'])
    
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=ax,
        cbar=False,
        xticklabels=range(len(np.unique(y_test))),
        yticklabels=range(len(np.unique(y_test))),
        square=True
    )
    ax.set_title(f'{model_name}', fontweight='bold', fontsize=12)
    ax.set_ylabel('Истинный класс')
    ax.set_xlabel('Предсказанный класс')

plt.tight_layout()
plt.show()

# 8. Выводы и рекомендации

**ОБЪЯСНЕНИЕ:**
Итоговый анализ результатов и рекомендации по выбору модели для конкретных задач.

In [ ]:
print("\n" + "="*70)
print("📋 ИТОГОВЫЙ АНАЛИЗ И ВЫВОДЫ")
print("="*70)

print(f"\n📊 ОБЩАЯ ИНФОРМАЦИЯ О ДАННЫХ:")
print("-" * 70)
print(f"  Фильмов: {len(df)}")
print(f"  Тем (классов): {len(np.unique(y))}")
for topic in range(len(np.unique(y))):
    count = (y == topic).sum()
    pct = count / len(y) * 100
    print(f"    Тема {topic}: {count:5d} фильмов ({pct:5.1f}%)")

print(f"\n📈 СТРУКТУРА ДАННЫХ ДЛЯ ОБУЧЕНИЯ:")
print("-" * 70)
print(f"  Всего признаков: {X_combined.shape[1]}")
print(f"    Числовые: {len(numeric_cols)}")
print(f"    Текстовые (TF-IDF): {tfidf_df.shape[1]}")
print(f"  Обучающая выборка: {X_train.shape[0]} образцов")
print(f"  Тестовая выборка: {X_test.shape[0]} образцов")

print(f"\n🏆 ЛУЧШИЕ МОДЕЛИ:")
print("-" * 70)
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    best_idx = results_df[metric].idxmax()
    best_model = results_df.loc[best_idx, 'Модель']
    best_value = results_df.loc[best_idx, metric]
    print(f"  По {metric:12s}: {best_model:30s} = {best_value:.4f}")

print(f"\n⏱️ ВРЕМЯ ОБУЧЕНИЯ:")
print("-" * 70)
for idx, row in results_df.iterrows():
    print(f"  {row['Модель']:30s}: {row['Время (сек)']:7.2f} сек")

print(f"\n" + "="*70)

# 9. Сохранение моделей

In [ ]:
# Сохраняем лучшую модель
print("\n" + "="*70)
print("💾 СОХРАНЕНИЕ МОДЕЛЕЙ")
print("="*70)

best_model_idx = results_df['F1-Score'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Модель']
best_model_obj = models_results[best_model_name]['model']

print(f"\nЛучшая модель по F1-Score: {best_model_name}")
print(f"  F1-Score: {results_df.loc[best_model_idx, 'F1-Score']:.4f}")

# Сохраняем лучшую модель
model_file = f'best_model_{best_model_name.replace(" ", "_")}.pkl'
with open(model_file, 'wb') as f:
    pickle.dump(best_model_obj, f)
print(f"✓ Модель сохранена в {model_file}")

# Сохраняем vectorizer
vectorizer_file = 'tfidf_vectorizer.pkl'
with open(vectorizer_file, 'wb') as f:
    pickle.dump(tfidf, f)
print(f"✓ TF-IDF vectorizer сохранен в {vectorizer_file}")

# Сохраняем scaler
scaler_file = 'minmax_scaler.pkl'
with open(scaler_file, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✓ MinMax scaler сохранен в {scaler_file}")

# Сохраняем результаты
results_df.to_csv('model_comparison_results.csv', index=False)
print(f"✓ Результаты сравнения сохранены в model_comparison_results.csv")

# Сохраняем темы и описания
topics_df = pd.DataFrame({
    'Topic': list(topics_description.keys()),
    'Top_Words': [', '.join(words) for words in topics_description.values()]
})
topics_df.to_csv('topics_nmf.csv', index=False)
print(f"✓ Описание тем сохранено в topics_nmf.csv")

print("\n✓ Все модели и артефакты успешно сохранены!")